# 🔧 Stage 2: Tool Calling Specialization
> Layer tool-calling capabilities on top of the Stage 1 code+reasoning model.

**Input Model:** Stage 1 merged model (or adapter on top of base)
**Method:** QLoRA with lower rank (tool calling is more focused)
**Dataset:** ~180K tool-calling samples (Hermes + Glaive + Gorilla)
**GPU Required:** A100/H100
**Estimated Time:** 4-6 hours (A100), 2-4 hours (H100)

### Why Stage 2 is Separate
Research shows hard distribution shifts cause catastrophic forgetting.
By training tool calling AFTER code+reasoning, we:
- Preserve the code generation skills from Stage 1
- Use a lower learning rate to avoid overwriting code knowledge
- Focus adapter capacity on the new tool-calling patterns


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
import torch
import json
import os
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"🖥️ GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Configuration — Stage 2
# Use Stage 1 output as the base, OR start from original + Stage 1 adapter
STAGE1_MODEL = "outputs/stage1_code_reasoning/merged_model"

# Fall back to base model if Stage 1 output not available
if not os.path.exists(STAGE1_MODEL):
    print("⚠️ Stage 1 model not found, using base Qwen model")
    print("   For best results, run notebook 02 first!")
    STAGE1_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

MAX_SEQ_LENGTH = 4096
LORA_RANK = 16        # Lower rank — tool calling is more focused than general code
LORA_ALPHA = 16
OUTPUT_DIR = "outputs/stage2_tool_calling"

# Lower LR to prevent catastrophic forgetting of Stage 1 skills
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 1e-4   # Half the Stage 1 LR — gentler updates
NUM_EPOCHS = 2
WARMUP_RATIO = 0.1     # Longer warmup for stability

print(f"✅ Using model: {STAGE1_MODEL}")

In [ ]:
# Load Stage 1 model
print("📥 Loading Stage 1 model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Add fresh LoRA adapters for Stage 2
print("🔧 Adding Stage 2 LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"📊 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Load Stage 2 dataset
DATA_PATH = "prepared_data/stage2_tool_calling.jsonl"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found. Run notebook 01 first.")

samples = []
with open(DATA_PATH, "r") as f:
    for line in f:
        samples.append(json.loads(line))

print(f"✅ Loaded {len(samples):,} tool-calling samples")

def format_for_training(sample):
    """Apply chat template, handling tool-calling specific roles."""
    messages = sample["messages"]
    # Filter out any empty messages
    messages = [m for m in messages if m.get("content", "").strip()]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}
    except Exception:
        # Fallback: manually format
        text = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        return {"text": text}

dataset = Dataset.from_list(samples)
dataset = dataset.map(format_for_training, remove_columns=dataset.column_names, num_proc=4)
# Filter out any empty/too-short examples
dataset = dataset.filter(lambda x: len(x["text"]) > 50)

print(f"✅ Formatted: {len(dataset):,} samples")

In [ ]:
# Train Stage 2
print("🏋️ Setting up Stage 2 trainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

print("🚀 Starting Stage 2 training...")
trainer_stats = trainer.train()

print(f"\n✅ Stage 2 complete!")
print(f"   Time: {trainer_stats.metrics['train_runtime']/3600:.1f} hours")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Save and merge
print("💾 Saving Stage 2 adapter...")
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print("🔀 Merging into final model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")

print(f"\n✅ Final model saved to {OUTPUT_DIR}/merged_model")
print("🎉 Proceed to notebook 04 for DPO, or notebook 05 for benchmarking!")